In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Load the Red Wine Quality dataset
# Ensure you have the dataset downloaded as 'winequality-red.csv'
data = pd.read_csv("winequality-red.csv", sep=';')  # The separator is ';'

# Separate features and target
X = data.drop('quality', axis=1)
y = data['quality']

# Encode the target variable (quality) if it's not already numerical
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Train a Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Step 4: Define the Integrated Gradients function
def integrated_gradients(model, input_data, baseline=None, steps=50):
    print("Input data:")
    print(input_data)
    
    # Use a zero baseline if none is provided
    if baseline is None:
        baseline = np.zeros_like(input_data)

    print("Baseline:")
    print(baseline)
    
    # Initialize an empty list to hold the interpolated inputs
    interpolated_inputs = []

    # Loop through each step from 0 to the number of steps (inclusive)
    for i in range(steps + 1):
        # Calculate the interpolation factor (i / steps)
        factor = i / steps
        
        # Calculate the interpolated point
        interpolated_point = baseline + factor * (input_data - baseline)
        
        # Append the interpolated point to the list
        interpolated_inputs.append(interpolated_point)

    # Convert the list of interpolated inputs to a NumPy array for easier manipulation
    interpolated_inputs = np.array(interpolated_inputs)

    # Predict model outputs for each interpolated input
    predictions = np.array([model.predict_proba([input])[0] for input in interpolated_inputs])

    # Get the predicted class for the original input
    target_class = np.argmax(model.predict_proba([input_data])[0])
    
    print("Target class:")
    print(target_class)
    
    # Calculate gradients by measuring change in prediction
    gradients = predictions[:, target_class] - predictions[0, target_class]
    
    print("Gradients:")
    print(gradients)
    
    # Average gradients across all steps and multiply by (input - baseline)
    avg_gradients = np.mean(gradients)
    
    print("Average gradients:")
    print(avg_gradients)
    
    integrated_grads = (input_data - baseline) * avg_gradients
    
    return integrated_grads

# Step 5: Choose an instance to explain (first test instance)
instance_to_explain = X_test.iloc[0].values  # Use .iloc to get the instance as an array

# Step 6: Compute Integrated Gradients for this instance
integrated_grads = integrated_gradients(model, instance_to_explain)

print("Integrated gradients:")
print(integrated_grads)

# Step 7: Display the feature importance based on Integrated Gradients
print("\nFeature importance (Integrated Gradients):")
for i, feature_name in enumerate(X.columns):
    print(f"{feature_name}: {integrated_grads[i]:.4f}")
